In [1]:
bt_enabled = True  # Set to False to stop

from pynq import Overlay, MMIO
from time import sleep

In [2]:
ol = Overlay("car.bit", ignore_version=True)
print("✔ Overlay 'car.bit' loaded successfully.\n")

✔ Overlay 'car.bit' loaded successfully.



In [3]:
# Access the AXI GPIO block (used for GPS enable/reset)
gpio = MMIO(ol.axi_gpio_0.mmio.base_addr, 0x10000, debug=False)

In [4]:
# AXI UARTLite base address
uart = MMIO(ol.Bluetooth.mmio.base_addr, 0x1000, debug=False)

print("✔ AXI UARTLite MMIO mapped")

✔ AXI UARTLite MMIO mapped


In [5]:
# Reset TX and RX FIFOs
uart.write(0x0C, 0x03)
sleep(0.1)

In [6]:
def uart_tx_byte(byte):
    # Wait until TX FIFO is not full
    while uart.read(0x08) & 0x08:
        pass
    uart.write(0x04, byte)

def uart_rx_byte():
    # Check if RX FIFO has data
    if uart.read(0x08) & 0x01:
        return uart.read(0x00) & 0xFF
    return None

def uart_send_string(s):
    for c in s:
        uart_tx_byte(ord(c))
        
def uart_read_all():
    resp = ""
    while True:
        b = uart_rx_byte()
        if b is None:
            break
        resp += chr(b)
    return resp

rx_buffer = ""

def uart_read_lines():
    global rx_buffer
    data = uart_read_all()
    if data:
        rx_buffer += data
        lines = rx_buffer.split("\n")
        rx_buffer = lines[-1]
        return lines[:-1]
    return []

for _ in range(100):
    lines = uart_read_lines()
    for line in lines:
        print("RX:", line)
    sleep(0.01)

print("Done")



Done


In [8]:
#Don't need to run again
uart_send_string("AT")
sleep(0.2)

response = ""
while True:
    b = uart_rx_byte()
    if b is None:
        break
    response += chr(b)

print("Bluetooth response:", response)

Bluetooth response: 


In [9]:
#Don't need to run again
uart_send_string("AT")

resp = ""
while True:
    b = uart_rx_byte()
    if b is None:
        break
    resp += chr(b)
    
print(resp)

In [9]:
#Don't need to run again
uart_send_string("AT+ROLE0")
sleep(0.2)

In [7]:
#Don't need to run
uart_send_string("AT+ROLE?")
sleep(0.2)

resp = ""
while True:
    b = uart_rx_byte()
    if b is None:
        break
    resp += chr(b)

print("ROLE response:", resp)

ROLE response: 


In [11]:
#Don't need to run
uart_send_string("AT+ADDR?")
sleep(0.2)

resp = ""
while True:
    b = uart_rx_byte()
    if b is None:
        break
    resp += chr(b)

print("Address response:", resp)

Address response: OK+ADDR:685E1C26


In [7]:
sleep(0.2)
print(uart_read_all())

In [8]:
for _ in range(1000):
    lines = uart_read_lines()
    for line in lines:
        print("RX:", line)
    sleep(0.01)

print("Done")

Done


In [9]:
sleep(0.1)
print(uart_read_all())

A


In [16]:
for _ in range(300):
    lines = uart_read_lines()
    for line in lines:
        print("RX:", line)
    sleep(0.01)

print("Done")

RX: Test, 123, 456, Is this working Is this working Is this working rrectly?
Done


In [14]:
sleep(0.1)
print(uart_read_all())

In [15]:
sleep(0.1)
print(uart_read_all())

In [19]:
for _ in range(1000):
    lines = uart_read_lines()
    for line in lines:
        print("RX:", line)
    sleep(0.01)

print("Done")

RX: Test, 123, 456, test test
RX: Is this working correctly?
RX: With numbers? I bench 280 and it is easy 58925 yeah Korean bbq(592, 829)
Done
